In [2]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from pyspark.context import SparkContext
from pyspark.sql.functions import explode, col, to_date
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame
from datetime import datetime
import boto3


# Initialize Glue
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session a5444dcc-cd77-48f7-8f20-b9b93feab2dd.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session a5444dcc-cd77-48f7-8f20-b9b93feab2dd.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 5.1


You are already connected to a glueetl session a5444dcc-cd77-48f7-8f20-b9b93feab2dd.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: None
Setting new worker type to: G.1X


You are already connected to a glueetl session a5444dcc-cd77-48f7-8f20-b9b93feab2dd.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: None
Setting new number of workers to: 5



In [17]:
partition_date = datetime.utcnow().strftime("%Y-%m-%d")

s3_path = (
    f"s3://spotify-etl-project-duc/"
    f"raw_data/to_processed/"
    f"extraction_date={partition_date}/"
)

In [25]:
source_dyf = glueContext.create_dynamic_frame_from_options(
    connection_type="s3",
    connection_options={"paths": [s3_path]},
    format="json"
)

spotify_df = source_dyf.toDF()

In [ ]:
if spotify_df.rdd.isEmpty():
    raise Exception(
        f"No data found in {s3_path}"
    )

In [29]:
def process_albums(df):
    df = (
        df
        .withColumn("items", explode("items"))
        .select(
            col("items.item.album.id").alias("album_id"),
            col("items.item.album.name").alias("album_name"),
            col("items.item.album.release_date").alias("release_date"),
            col("items.item.album.total_tracks").alias("total_tracks"),
            col("items.item.album.external_urls.spotify").alias("url")
        )
        .dropDuplicates(["album_id"])
        .withColumn(
            "release_date",
            to_date("release_date")
        )
    )

    return df

In [30]:
def process_artists(df):
    return (
        df
        .select(
            explode("items").alias("item")
        )
        .select(
            explode(
                "item.item.artists"
            ).alias("artist")
        )
        .select(
            col("artist.id").alias("artist_id"),
            col("artist.name").alias("artist_name"),
            col("artist.external_urls.spotify").alias("external_url")
        )
        .dropDuplicates(["artist_id"])
    )

In [31]:
def process_songs(df):
    df_exploded = df.select(
        explode(col("items")).alias("item")
    )

    df_songs = (
        df_exploded
        .select(
            col("item.item.id").alias("song_id"),
            col("item.item.name").alias("song_name"),
            col("item.item.duration_ms").alias("song_duration_ms"),
            col("item.item.external_urls.spotify").alias("song_url"),
            col("item.item.explicit").alias("explicit"),
            col("item.added_at").alias("song_added"),
            col("item.item.album.id").alias("album_id"),
            col("item.item.artists")[0]["id"].alias("artist_id")
        )
        .dropDuplicates(["song_id"])
    )

    df_songs = df_songs.withColumn(
        "song_added",
        to_date(col("song_added"))
    )

    return df_songs

In [32]:
album_df = process_albums(spotify_df)
artist_df = process_artists(spotify_df)
song_df = process_songs(spotify_df)

In [37]:
def write_to_s3(
    df,
    table_name,
    partition_date,
    format_type="parquet"
):
    dynamic_frame = DynamicFrame.fromDF(
        df,
        glueContext,
        f"{table_name}_dynamic_frame"
    )

    glueContext.write_dynamic_frame.from_options(
        frame=dynamic_frame,
        connection_type="s3",
        connection_options={
            "path": (
                f"s3://spotify-etl-project-duc/"
                f"transformed_data/"
                f"{table_name}/"
                f"extraction_date={partition_date}/"
            )
        },
        format=format_type
    )

In [39]:
write_to_s3(album_df, "album", partition_date)
write_to_s3(artist_df, "artist", partition_date)
write_to_s3(song_df, "songs", partition_date)

In [ ]:
def list_s3_objects(bucket, prefix):
    s3_client = boto3.client("s3")

    response = s3_client.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix
    )

    return [
        content["Key"]
        for content in response.get("Contents", [])
        if content["Key"].endswith(".json")
    ]


def move_and_delete_files(spotify_keys, bucket):
    s3_resource = boto3.resource("s3")

    for key in spotify_keys:
        copy_source = {
            "Bucket": bucket,
            "Key": key
        }

        # Preserve extraction_date partition
        destination_key = key.replace(
            "raw_data/to_processed/",
            "raw_data/processed/"
        )

        print(f"Moving {key} -> {destination_key}")

        # Copy file
        s3_resource.meta.client.copy(
            copy_source,
            bucket,
            destination_key
        )

        # Delete original file
        s3_resource.Object(
            bucket,
            key
        ).delete()

In [39]:
bucket_name = "spotify-etl-project-duc"

prefix = (
    "raw_data/to_processed/"
    f"extraction_date={partition_date}/"
)

spotify_keys = list_s3_objects(
    bucket_name,
    prefix
)

move_and_delete_files(
    spotify_keys,
    bucket_name
)